In [11]:
import pandas as pd

train_df = pd.read_pickle('train_df_without_task3_labels.pkl')
test_df = pd.read_pickle('test_df_without_task3_labels.pkl')
dev_df = pd.read_pickle('dev_df_without_task3_labels.pkl')


In [12]:
import re
from collections import Counter
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import csv

from nltk.tokenize import word_tokenize

from torch.utils.data import Dataset, DataLoader

class BiLSTM(nn.Module):
    def __init__(self,vocab_size, embedding_dim, hidden_dim, output_dim):
        super(BiLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        hidden = torch.cat((output[:, -1, :self.hidden_dim], output[:, 0, self.hidden_dim:]), dim=1)
        return self.fc(hidden)

def preprocess(text):
#   text = re.sub(r'[^0-9a-zA-Z\s]', ' ', text)
#   text = text.lower()
    tokens = word_tokenize(text)
#   tokens = [token for token in tokens if token not in stop_words] 
    return tokens
#   return ["<start>"] +  tokens + ["<end>"]

class SentenceDataset(Dataset):
    def __init__(self, df):
        self.sentences = df["padded_sentence"].tolist()
        self.labels = df["class_idx"].tolist()
        # print(self.labels)

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return torch.tensor(self.sentences[idx]), torch.tensor(self.labels[idx])


In [13]:
train_data = train_df.copy()
raw_sentences = train_data['text']
raw_keyphrases = train_data['phrase']

train_data['Class Index'] = train_data['label'].replace({
    'Task': 0,
    'Process': 1,
    'Material': 2
})

labels = train_data['Class Index']

max_length = max(len(word_tokenize(sentence)) for sentence in raw_sentences)
max_length_key = max(len(word_tokenize(keyphrase)) for keyphrase in raw_keyphrases) 

sentences = []
sentences_tokenised = []
keyphrases = []
keyphrases_tokenised = []

for keyphrase, sentence in zip(raw_keyphrases,raw_sentences):
    # print
    tokenised_sentence = preprocess(sentence)
    tokenised_keyphrase = preprocess(keyphrase)
    sentences.append(" ".join(tokenised_sentence))
    keyphrases.append(" ".join(tokenised_keyphrase))
    sentences_tokenised.append((tokenised_sentence))
    keyphrases_tokenised.append((tokenised_keyphrase))

print(len(sentences))
train_data['sentence'] = sentences
train_data['keyphrase'] = keyphrases
train_data['class_idx'] = labels

padded_sentences = []
padded_keyphrase = []
comb = []
# max_length = 100
for keyphrase, sentence in zip(keyphrases_tokenised, sentences_tokenised):
    padded_sentence = sentence + ["<PAD>"] * (max_length - len(sentence))
    padded_keyphrase = keyphrase + ["<PAD>"] * (max_length_key - len(keyphrase))
    concat = padded_keyphrase[0:max_length_key] + padded_sentence[:max_length]
    padded_sentences.append(" ".join(padded_sentence))
    padded_keyphrase.append(" ".join(padded_keyphrase))
    comb.append(" ".join(concat))

# train_data['padded_sentence'] = padded_sentences
train_data['padded_sentence'] = comb
train_data['combined'] = comb


6706


In [14]:
# def main(dev_df.copy()):
dev_data = dev_df.copy()
raw_sentences = dev_data['text']
raw_keyphrases = dev_data['phrase']

dev_data['Class Index'] = dev_data['label'].replace({
    'Task': 0,
    'Process': 1,
    'Material': 2
})

labels = dev_data['Class Index']

sentences = []
sentences_tokenised = []
keyphrases = []
keyphrases_tokenised = []

for keyphrase, sentence in zip(raw_keyphrases,raw_sentences):
    # print
    tokenised_sentence = preprocess(sentence)
    tokenised_keyphrase = preprocess(keyphrase)
    sentences.append(" ".join(tokenised_sentence))
    keyphrases.append(" ".join(tokenised_keyphrase))
    sentences_tokenised.append((tokenised_sentence))
    keyphrases_tokenised.append((tokenised_keyphrase))

print(len(sentences))
dev_data['sentence'] = sentences
dev_data['keyphrase'] = keyphrases
dev_data['class_idx'] = labels

padded_sentences = []
padded_keyphrase = []
comb = []
# max_length = 100
for keyphrase, sentence in zip(keyphrases_tokenised, sentences_tokenised):
    padded_sentence = sentence + ["<PAD>"] * (max_length - len(sentence))
    padded_keyphrase = keyphrase + ["<PAD>"] * (max_length_key - len(keyphrase))
    concat = padded_keyphrase[0:max_length_key] + padded_sentence[:max_length]
    padded_sentences.append(" ".join(padded_sentence))
    padded_keyphrase.append(" ".join(padded_keyphrase))
    comb.append(" ".join(concat))

# train_data['padded_sentence'] = padded_sentences
dev_data['padded_sentence'] = comb
dev_data['combined'] = comb


1129


In [15]:
# def main(dev_df.copy()):
test_data = test_df.copy()
raw_sentences = test_data['text']
raw_keyphrases = test_data['phrase']

test_data['Class Index'] = test_data['label'].replace({
    'Task': 0,
    'Process': 1,
    'Material': 2
})

labels = test_data['Class Index']

sentences = []
sentences_tokenised = []
keyphrases = []
keyphrases_tokenised = []

for keyphrase, sentence in zip(raw_keyphrases,raw_sentences):
    # print
    tokenised_sentence = preprocess(sentence)
    tokenised_keyphrase = preprocess(keyphrase)
    sentences.append(" ".join(tokenised_sentence))
    keyphrases.append(" ".join(tokenised_keyphrase))
    sentences_tokenised.append((tokenised_sentence))
    keyphrases_tokenised.append((tokenised_keyphrase))

print(len(sentences))
test_data['sentence'] = sentences
test_data['keyphrase'] = keyphrases
test_data['class_idx'] = labels

padded_sentences = []
padded_keyphrase = []
comb = []
# max_length = 100z
for keyphrase, sentence in zip(keyphrases_tokenised, sentences_tokenised):
    padded_sentence = sentence + ["<PAD>"] * (max_length - len(sentence))
    padded_keyphrase = keyphrase + ["<PAD>"] * (max_length_key - len(keyphrase))
    concat = padded_keyphrase[0:max_length_key] + padded_sentence[:max_length]
    padded_sentences.append(" ".join(padded_sentence))
    padded_keyphrase.append(" ".join(padded_keyphrase))
    comb.append(" ".join(concat))

# train_data['padded_sentence'] = padded_sentences
test_data['padded_sentence'] = comb
test_data['combined'] = comb


2052


In [16]:

train_df = train_data.copy()
test_df = test_data.copy()
dev_df = dev_data.copy()

words = [word for sentence in train_df["padded_sentence"] for word in sentence.split()]
vocab = {word: i for i, (word, _) in enumerate(Counter(words).items())}
vocab_size = len(vocab)

train_df["padded_sentence"] = train_df["padded_sentence"].apply(lambda x: [vocab[word] for word in x.split()])
dev_df["padded_sentence"] = dev_df["padded_sentence"].apply(lambda x: [vocab.get(word, 0) for word in x.split()])
test_df["padded_sentence"] = test_df["padded_sentence"].apply(lambda x: [vocab.get(word, 0) for word in x.split()])
batch_size = 64

train_dataset = SentenceDataset(train_df)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

dev_dataset = SentenceDataset(dev_df)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size)

test_dataset = SentenceDataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


embedding_dim = 100
hidden_dim = 128
output_dim = 3
learning_rate = 0.001
num_epochs = 5

model = BiLSTM(vocab_size,embedding_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)



In [20]:

print("training")
model.train()
for epoch in range(5):
    i = 0
    for sentences, labels in train_loader:
        # print(sentences, "jj", labels)
        if i % len(train_loader)/10 == 0:
            print(i, len(train_loader))
        i += 1
        optimizer.zero_grad()
        predictions = model(sentences)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

    model.eval()
    correct = 0
    total = 0

    predictions_list = []
    correct_list = []
    with torch.no_grad():
        for sentences, labels in dev_loader:
            predictions = model(sentences)
            _, predicted = torch.max(predictions.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            predictions_list.extend(predicted.cpu().numpy())
            correct_list.extend(labels.cpu().numpy())
    print(f"Accuracy: {correct/total}")
    
with open(f'dev_predictions.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['predicted_label', 'correct_label'])
    for predicted_label, correct_label in zip(predictions_list, correct_list):
        writer.writerow([predicted_label, correct_label])



training
0 105
Epoch 1, Loss: 0.01498143095523119
Accuracy: 0.728963684676705
0 105
Epoch 2, Loss: 0.004767229314893484
Accuracy: 0.7298494242692648
0 105
Epoch 3, Loss: 0.005045056343078613
Accuracy: 0.7254207263064659
0 105
Epoch 4, Loss: 0.0019490026170387864
Accuracy: 0.7077059344552702
0 105


In [ ]:
model.eval()
correct = 0
total = 0

predictions_list = []
correct_list = []
with torch.no_grad():
    for sentences, labels in test_loader:
        predictions = model(sentences)
        _, predicted = torch.max(predictions.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        predictions_list.extend(predicted.cpu().numpy())
        correct_list.extend(labels.cpu().numpy())
print(f"Accuracy: {correct/total}")

with open(f'test_predictions.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['predicted_label', 'correct_label'])
    for predicted_label, correct_label in zip(predictions_list, correct_list):
        writer.writerow([predicted_label, correct_label])